In [6]:
# import library
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re
import string
import emoji
import nltk

nltk.download('stopwords')
nltk.download('punkt')
indonesian_stopwords = set(stopwords.words('indonesian'))
stemmer = PorterStemmer()

### Load dataset

In [7]:
df = pd.read_csv("reviews.csv")
print("Dataset Head:\n", df.head())

# Pastikan kolom sesuai
assert 'reviews' in df.columns, "Kolom 'reviews' tidak ditemukan dalam dataset"
assert 'label' in df.columns, "Kolom 'label' tidak ditemukan dalam dataset"

Dataset Head:
                                              reviews  label
0  kemeja nya bagusss bgtttt😍😍😍aaaa mauuu nngisss...    1.0
1  Jahitannya sih rapi,cuman ada benang yang ikut...    0.0
2  Sesuai harga. Agak tipis tapi masih oke kok. W...    0.0
3  Wah gila sihhh sebagus itu, se worth it, se  l...    1.0
4  Kain nya bagus halus  \nTapi kok di bukak koto...    0.0


### Preprocessing teks

In [ ]:
def clean_text(text):
    text = emoji.replace_emoji(text, '') # Hapus emoji
    text = text.lower()  # Lowercasing
    text = re.sub(r'\d+', '', text)  # Menghapus angka
    text = text.translate(str.maketrans("", "", string.punctuation))  # Menghapus tanda baca
    text = word_tokenize(text)  # Tokenisasi / split word
    text = [word for word in text if word not in indonesian_stopwords]  # Menghapus stopwords / di ke dll
    text = [stemmer.stem(word) for word in text]  # Stemming / ke bentuk dasar
    return " ".join(text)

df['text_clean'] = df['reviews'].apply(clean_text)
print("Cleaned Text Samples:\n", df[['reviews', 'text_clean']].head())

Cleaned Text Samples:
                                              reviews  \
0  kemeja nya bagusss bgtttt😍😍😍aaaa mauuu nngisss...   
1  Jahitannya sih rapi,cuman ada benang yang ikut...   
2  Sesuai harga. Agak tipis tapi masih oke kok. W...   
3  Wah gila sihhh sebagus itu, se worth it, se  l...   
4  Kain nya bagus halus  \nTapi kok di bukak koto...   

                                          text_clean  
0  kemeja nya bagusss bgttttaaaa mauuu nngisssssk...  
1        jahitannya sih rapicuman benang jahit jelek  
2  sesuai harga tipi oke warnanya abu kalo difoto...  
3  gila sihhh sebagu worth it lembut bajunya kira...  
4      kain nya bagu halu bukak kotor ya warna putih  


### Split dataset training & testing

In [ ]:
# Mengecek distribusi label dalam dataset
label_counts = df['label'].value_counts()
print("Distribusi Label dalam Dataset:")
print(label_counts)
print("\nPersentase Kelas Positif & Negatif:")
print(label_counts / label_counts.sum() * 100)

# trains
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text_clean'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)
print("Training Set Size:", len(train_texts))
print("Testing Set Size:", len(test_texts))

Training Set Size: 664
Testing Set Size: 167


### Tokenisasi 

In [ ]:
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p2") #  pre-trained model IndoBERT.
print("Tokenizer Loaded.")

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer)
print("Datasets Created.")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
print("Dataloaders Created.")

Tokenizer Loaded.
Datasets Created.
Dataloaders Created.


### Fine-tuning

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p2", num_labels=2)
model.to(device)
print("Model Loaded and Moved to Device.")

optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

def train_model(model, train_loader, optimizer, criterion, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}")

train_model(model, train_loader, optimizer, criterion)

Model Loaded and Moved to Device.
Epoch 1, Loss: 0.37126942684075664


Epoch 2, Loss: 0.08540220740472987
Epoch 3, Loss: 0.04745032294609007


### Evaluasi

In [12]:
def evaluate_model(model, test_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(true_labels, predictions)
    print(f"Accuracy: {accuracy:.4f}")
    print(classification_report(true_labels, predictions))

evaluate_model(model, test_loader)

Accuracy: 0.9401
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        84
           1       0.93      0.95      0.94        83

    accuracy                           0.94       167
   macro avg       0.94      0.94      0.94       167
weighted avg       0.94      0.94      0.94       167

